# Hello World

In [ ]:
print('Hello, beautiful world!')

# Dataset Manipulation

## Function def

In [ ]:
import os
import random
from PIL import Image
import matplotlib.pyplot as plt

def display_random_images(folder_path, num_images=5):

    png_files = [f for f in os.listdir(folder_path)]

    print(f"Total number of PNG images in the folder: {len(png_files)}")

    selected_images = random.sample(png_files, num_images)

    plt.figure(figsize=(10, 5))  
    for i, image_name in enumerate(selected_images, 1):
        image_path = os.path.join(folder_path, image_name)
        image = Image.open(image_path)

        
        plt.subplot(2, 5, i)  
        plt.imshow(image)
        plt.title(image_name)
        plt.axis('off')  

    plt.tight_layout()
    plt.show()


## Image display

In [ ]:

folder_path = 'C:/Users/mohanned/Downloads/Dots Scattered by Halton Sequence.mglyph (Unzipped Files)-20250227T083919Z-001/Dots Scattered by Halton Sequence.mglyph (Unzipped Files)' 
display_random_images(folder_path)

# Stars Dataset Visualization

## Mounting google drive

In [ ]:
%pip install pydrive 

In [ ]:
%pip install --upgrade google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [ ]:
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle
import os

SCOPES = ['https://www.googleapis.com/auth/drive']

creds = None
if os.path.exists('token.pickle'):
    with open('token.pickle', 'rb') as token:
        creds = pickle.load(token)
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
    with open('token.pickle', 'wb') as token:
        pickle.dump(creds, token)

drive_service = build('drive', 'v3', credentials=creds)


results = drive_service.files().list(pageSize=10, fields="files(id, name)").execute()
files = results.get('files', [])
for file in files:
    print(f"Found file: {file['name']} (ID: {file['id']})")


Found file: Sujets PFAI INFO & TEL  (ID: 1QvDeub4TtEKpYEZu0o4arwdCkg15zkdefZiyasGG6SU)
Found file: Random Colored Star 2.mglyph (Unzipped Files) (ID: 1Wi2nG9SOblXBjNP_YQ5i-wsB3cYZlL97)
Found file: Copy of Glyph_Manip.ipynb (ID: 1OL40pb1OgnZIRxb_PYTWP9HYWGSoCYL7)
Found file: Random Colored Star 30.mglyph (ID: 10ui0TIQnRxz_iXVsJrdM_gxp4kIqPlvA)
Found file: Random Colored Star 29.mglyph (ID: 10pPs9HcMZeVCoo5tRZWTXFCOPO0IbP_c)
Found file: Random Colored Star 28.mglyph (ID: 10p9JO58ph6dN_go9FKvezTXqW4nyBg9Q)
Found file: Random Colored Star 27.mglyph (ID: 10oJ4xIgAEQrRcODchHPmwSMfbky7XW4V)
Found file: Random Colored Star 26.mglyph (ID: 10k5JHJ0_B_ABBNCCmc0KhNdFXi-8IQzV)
Found file: Random Colored Star 25.mglyph (ID: 10jx3-VzWmnB2psx2EiPOkKxmBP7_UkAA)
Found file: Random Colored Star 24.mglyph (ID: 10g9Rv2wQLophjMyQbKyQIlxwXTTVIsBu)


In [ ]:
import io
import os
import zipfile
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload


parent_folder_id = '1-9d5iNTJPv9VWla-j4PcCcza6igslBoj'


drive_service = build('drive', 'v3')


query = f"'{parent_folder_id}' in parents and name contains '.mglyph' and mimeType='application/zip'"
results = drive_service.files().list(q=query, fields="files(id, name)").execute()
files = results.get('files', [])

if not files:
    print("No .mglyph zip files found.")
else:
    for file in files:
        file_id = file['id']
        file_name = file['name']
        folder_name = file_name.replace('.mglyph', '')  # Extract folder name

        print(f"Processing: {file_name}")

        
        request = drive_service.files().get_media(fileId=file_id)
        file_stream = io.BytesIO()
        downloader = MediaIoBaseDownload(file_stream, request)
        
        done = False
        while not done:
            _, done = downloader.next_chunk()

        
        file_stream.seek(0)

        
        query = f"'{parent_folder_id}' in parents and name='{folder_name}' and mimeType='application/vnd.google-apps.folder'"
        existing_folder = drive_service.files().list(q=query, fields="files(id)").execute().get('files', [])
        
        if existing_folder:
            extracted_folder_id = existing_folder[0]['id']
            print(f"Folder '{folder_name}' already exists.")
        else:
            
            folder_metadata = {
                'name': folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [parent_folder_id]
            }
            folder = drive_service.files().create(body=folder_metadata, fields="id").execute()
            extracted_folder_id = folder['id']
            print(f"Created folder: {folder_name}")

        
        with zipfile.ZipFile(file_stream, 'r') as zip_ref:
            for extracted_file in zip_ref.namelist():
                extracted_content = zip_ref.read(extracted_file)  
                
                
                file_metadata = {
                    'name': extracted_file,
                    'parents': [extracted_folder_id]
                }
                media = MediaIoBaseUpload(io.BytesIO(extracted_content), mimetype='application/octet-stream')
                drive_service.files().create(body=file_metadata, media_body=media).execute()
                
                print(f"Extracted and uploaded: {extracted_file} to {folder_name}")
